# 🛣️ Pure Prompt-on-Demand Road Damage & Infrastructure Segmentation
### Murni Menggunakan SAM (Segment Anything Model 2.1 / SAM 3.1) Tanpa YOLO

Notebook ini mengimplementasikan segmentasi permukaan jalan dan infrastruktur (**Pothole, Manhole, Traffic Sign, Road Crack**) secara **Murni (Pure) SAM Prompt-on-Demand** langsung menggunakan kemampuan spasial dan konsep model SAM tanpa ketergantungan pada model detektor eksternal (YOLOv8 / YOLO-World).

---

### 🌟 Fitur Utama:
1. **100% Pure SAM Inference**:
   - Menggunakan arsitektur terkini **SAM 2.1** (`sam2.1_t.pt` / `sam2.1_s.pt`) dari Meta AI dengan akselerasi GPU CUDA (RTX 3050).
   - Mendukung peralihan otomatis ke **SAM 3** jika file checkpoint `sam3.pt` diletakkan di direktori kerja.
2. **Prompt-on-Demand Spasial Fleksibel**:
   - **Box Prompting (`bboxes`)**: Menentukan area target langsung pada lubang jalan, retakan, tutup got, atau rambu.
   - **Point Prompting (`points` & `labels`)**: Menentukan titik koordinat target (foreground: 1) dan titik pengecualian (background: 0).
   - **Auto-Segment Everything**: Pemindaian otomatis seluruh segmen tekstur permukaan jalan secara zero-shot.
3. **Pengujian Prompt Terpisah (Individual Testing)**:
   - Pengujian mandiri sel demi sel untuk masing-masing prompt: **`pothole`**, **`manhole`**, **`sign`**, dan **`crack`**.
4. **Unified Multi-Prompt Pipeline**:
   - Seluruh prompt digabungkan dalam satu visualisasi lengkap dengan palet warna terpisah dan kalkulasi metrik luas kerusakan jalan (*damage severity metrics*).
5. **Inferensi Video (`testvideo1.mp4`) & Placeholder Kelas Dinamis**:
   - Dilengkapi dictionary konfigurasi placeholder yang dapat ditambah, dikurangi, atau diubah kelasnya sesuka hati untuk mendeteksi apa saja sesuai kebutuhan riset Hibah VLM.
   - Menyimpan hasil video ke `hasil_video_pure_sam.mp4` lengkap dengan HUD telemetri real-time.

## 1. Setup Environment & Inisialisasi Dependensi
Memuat pustaka PyTorch, OpenCV, Matplotlib, serta pustaka SAM dari Ultralytics (Murni SAM tanpa YOLO).

In [ ]:
import os
from pathlib import Path
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import SAM

print(f"Ultralytics version : {ultralytics.__version__}")
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total           : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    device = 0
else:
    print("⚠️ CUDA tidak terdeteksi, inferensi berjalan di CPU.")
    device = "cpu"


## 2. Load Model SAM 2.1 / SAM 3.1 (Pure SAM)

Kita memuat model SAM secara langsung. Checkpoint `sam2.1_t.pt` (SAM 2.1 Tiny) telah teroptimasi dengan sangat baik untuk GPU RTX 3050 (hemat VRAM ~1.5 GB dan berkecepatan tinggi). Jika file `sam3.pt` ada di folder, sistem akan otomatis menggunakannya.

In [ ]:
# Pemuatan Model SAM (Adaptif: SAM 3 jika sam3.pt ada, atau SAM 2.1 Tiny)
sam_checkpoint = "sam3.pt" if Path("sam3.pt").exists() else "sam2.1_t.pt"
print(f"Memuat Pure SAM Model dari: {sam_checkpoint}...")
sam_model = SAM(sam_checkpoint)
print("✅ Pure SAM Model berhasil dimuat dan siap untuk Prompt-on-Demand!")


## 3. Fungsi Bantuan Visualisasi & Eksekusi Pure SAM Prompt-on-Demand

Fungsi di bawah menangani inferensi murni SAM menggunakan prompt spasial (Bounding Box atau Point Koordinat) tanpa memerlukan detektor YOLO.

In [ ]:
def run_pure_sam_prompt(
    image_source,
    sam_net,
    bboxes=None,
    points=None,
    labels=None,
    color=(0, 0, 255),
    prompt_name="Object",
    device=0
):
    """
    Menjalankan inferensi murni SAM berbasis Box Prompt dan/atau Point Prompt.
    - bboxes: list kotak pembatas [[x1, y1, x2, y2], ...]
    - points: list titik koordinat [[x, y], ...]
    - labels: list label titik [1, ...] (1 = foreground, 0 = background)
    """
    if isinstance(image_source, (str, Path)):
        img = cv2.imread(str(image_source))
        if img is None:
            raise IOError(f"Gambar tidak ditemukan: {image_source}")
    else:
        img = image_source.copy()

    h, w = img.shape[:2]
    overlay = img.copy()
    detection_records = []

    # Validasi input prompt
    has_boxes = bboxes is not None and len(bboxes) > 0
    has_points = points is not None and len(points) > 0

    if not has_boxes and not has_points:
        print("⚠️ Tidak ada prompt yang diberikan ke SAM.")
        return overlay, detection_records, None

    # Normalisasi format input
    sam_boxes = bboxes if has_boxes else None
    sam_points = points if has_points else None
    sam_labels = labels if (has_points and labels is not None) else ([1] * len(points) if has_points else None)

    # Eksekusi Pure SAM
    sam_res = sam_net.predict(
        source=img,
        bboxes=sam_boxes,
        points=sam_points,
        labels=sam_labels,
        retina_masks=True,
        verbose=False,
        device=device
    )[0]

    if sam_res.masks is not None:
        masks_np = sam_res.masks.data.cpu().numpy()
        num_masks = len(masks_np)

        for i in range(num_masks):
            mask_i = (masks_np[i] > 0.5).astype(np.uint8)
            if mask_i.shape != (h, w):
                mask_i = cv2.resize(mask_i, (w, h), interpolation=cv2.INTER_NEAREST)

            # 1. Overlay Mask Transparan
            colored_mask = np.zeros_like(img, dtype=np.uint8)
            colored_mask[mask_i == 1] = color
            overlay = cv2.addWeighted(overlay, 1.0, colored_mask, 0.45, 0)

            # 2. Garis Kontur Halus
            contours, _ = cv2.findContours(mask_i, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay, contours, -1, color, 2)

            # 3. Render visual Box jika ada
            if has_boxes and i < len(bboxes):
                bx1, by1, bx2, by2 = map(int, bboxes[i])
                cv2.rectangle(overlay, (bx1, by1), (bx2, by2), color, 2)
                lbl_text = f"{prompt_name} [Box {i+1}]"
                cv2.putText(overlay, lbl_text, (bx1, max(20, by1 - 8)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

            # 4. Render visual Point jika ada
            if has_points and i < len(points):
                px, py = map(int, points[i])
                cv2.circle(overlay, (px, py), 6, (0, 255, 255), -1)
                cv2.circle(overlay, (px, py), 8, (0, 0, 0), 2)
                lbl_text = f"{prompt_name} [Pt {i+1}]"
                cv2.putText(overlay, lbl_text, (px + 10, py - 5), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

            # Hitung statistik area piksel
            pixel_area = int(np.sum(mask_i))
            area_pct = (pixel_area / (h * w)) * 100.0

            detection_records.append({
                "prompt": prompt_name,
                "mask_id": i + 1,
                "pixel_area": pixel_area,
                "area_percent": area_pct
            })

    return overlay, detection_records, sam_res

def show_coordinate_helper(image_path, title="Panduan Koordinat Gambar"):
    """
    Menampilkan gambar dengan sumbu koordinat piksel untuk mempermudah
    menentukan koordinat Box / Point prompt secara presisi.
    """
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        return
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    plt.figure(figsize=(10, 6))
    plt.imshow(img_rgb)
    plt.title(f"{title} (Lebar: {w}px, Tinggi: {h}px)", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.4, color="yellow")
    plt.xlabel("Koordinat X (piksel)", fontsize=10)
    plt.ylabel("Koordinat Y (piksel)", fontsize=10)
    plt.show()

print("✅ Fungsi 'run_pure_sam_prompt' dan 'show_coordinate_helper' siap digunakan!")


### 📍 Panduan Menentukan Titik/Kotak Koordinat pada Gambar Uji
Gunakan tampilan bergrid di bawah ini untuk melihat posisi piksel objek (pothole, manhole, crack, sign) yang ingin Anda jadikan prompt.

In [ ]:
# Tampilkan panduan koordinat pada gambar jalan 'testimage3.png'
show_coordinate_helper("testimage3.png", "Koordinat Piksel testimage3.png")


## 4. Pengujian Prompt Terpisah (Individual Prompt Testing)

Sesuai permintaan riset, pada bagian ini kita menguji masing-masing kategori prompt secara **terpisah** langsung menggunakan Pure SAM:
- **Prompt A:** `pothole` (Lubang Permukaan Jalan)
- **Prompt B:** `manhole` (Tutup Gorong-gorong/Got)
- **Prompt C:** `sign` / `traffic sign` (Rambu Jalan)
- **Prompt D:** `crack` / `road crack` (Retakan Aspal)

In [ ]:
# ==============================================================================
# 🕳️ PENGUJIAN PROMPT TERPISAH 1: POTHOLE (LUBANG JALAN)
# ==============================================================================
TEST_IMG = "testimage3.png"

# Prompt On-Demand untuk Pothole:
# 1. Box Prompt pada area lubang besar di kanan bawah [x1, y1, x2, y2]
# 2. Point Prompt tepat di tengah lubang aspal [x, y]
POTHOLE_BOXES = [[424, 323, 720, 454]]
POTHOLE_POINTS = [[560, 385]]

print("Menjalankan Pure SAM untuk prompt: 'Pothole (Lubang Jalan)'...")
overlay_pothole, dets_pothole, _ = run_pure_sam_prompt(
    image_source=TEST_IMG,
    sam_net=sam_model,
    bboxes=POTHOLE_BOXES,
    points=POTHOLE_POINTS,
    color=(0, 0, 255),      # Merah (BGR)
    prompt_name="Pothole"
)

# Visualisasi Side-by-Side
img_orig = cv2.cvtColor(cv2.imread(TEST_IMG), cv2.COLOR_BGR2RGB)
res_rgb = cv2.cvtColor(overlay_pothole, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_orig)
axes[0].set_title(f"Gambar Asli: {TEST_IMG}", fontsize=12)
axes[0].axis("off")

axes[1].imshow(res_rgb)
axes[1].set_title(f"Hasil Segmentasi Pure SAM: Pothole ({len(dets_pothole)} segmen)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("📊 Ringkasan Metrik Pothole:")
for d in dets_pothole:
    print(f"  • Segmen {d['mask_id']}: Luas {d['pixel_area']:,} px ({d['area_percent']:.2f}% dari luas gambar)")


In [ ]:
# ==============================================================================
# 🕳️ PENGUJIAN PROMPT TERPISAH 2: MANHOLE (TUTUP GORONG-GORONG / SALURAN AIR)
# ==============================================================================
TEST_IMG = "testimage3.png"

# Prompt On-Demand untuk Manhole:
# Tutup got berada di sebelah kiri tengah
MANHOLE_BOXES = [[78, 376, 179, 414]]
MANHOLE_POINTS = [[128, 395]]

print("Menjalankan Pure SAM untuk prompt: 'Manhole (Tutup Got)'...")
overlay_manhole, dets_manhole, _ = run_pure_sam_prompt(
    image_source=TEST_IMG,
    sam_net=sam_model,
    bboxes=MANHOLE_BOXES,
    points=MANHOLE_POINTS,
    color=(0, 255, 0),      # Hijau (BGR)
    prompt_name="Manhole"
)

# Visualisasi Side-by-Side
res_rgb = cv2.cvtColor(overlay_manhole, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_orig)
axes[0].set_title(f"Gambar Asli: {TEST_IMG}", fontsize=12)
axes[0].axis("off")

axes[1].imshow(res_rgb)
axes[1].set_title(f"Hasil Segmentasi Pure SAM: Manhole ({len(dets_manhole)} segmen)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("📊 Ringkasan Metrik Manhole:")
for d in dets_manhole:
    print(f"  • Segmen {d['mask_id']}: Luas {d['pixel_area']:,} px ({d['area_percent']:.2f}% dari luas gambar)")


In [ ]:
# ==============================================================================
# 🛑 PENGUJIAN PROMPT TERPISAH 3: SIGN / TRAFFIC SIGN (RAMBU LALU LINTAS)
# ==============================================================================
TEST_IMG_SIGN = "testimage4.png" if Path("testimage4.png").exists() else "testimage3.png"

# Tampilkan panduan koordinat gambar rambu
show_coordinate_helper(TEST_IMG_SIGN, f"Panduan Koordinat {TEST_IMG_SIGN}")

# Box Prompt & Point Prompt pada rambu lalu lintas
SIGN_BOXES = [[450, 80, 750, 420]]
SIGN_POINTS = [[600, 250]]

print(f"Menjalankan Pure SAM untuk prompt: 'Traffic Sign' pada {TEST_IMG_SIGN}...")
overlay_sign, dets_sign, _ = run_pure_sam_prompt(
    image_source=TEST_IMG_SIGN,
    sam_net=sam_model,
    bboxes=SIGN_BOXES,
    points=SIGN_POINTS,
    color=(0, 255, 255),    # Kuning (BGR)
    prompt_name="Sign"
)

# Visualisasi Side-by-Side
img_orig_sign = cv2.cvtColor(cv2.imread(TEST_IMG_SIGN), cv2.COLOR_BGR2RGB)
res_rgb = cv2.cvtColor(overlay_sign, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_orig_sign)
axes[0].set_title(f"Gambar Asli: {TEST_IMG_SIGN}", fontsize=12)
axes[0].axis("off")

axes[1].imshow(res_rgb)
axes[1].set_title(f"Hasil Segmentasi Pure SAM: Sign ({len(dets_sign)} segmen)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("📊 Ringkasan Metrik Sign:")
for d in dets_sign:
    print(f"  • Segmen {d['mask_id']}: Luas {d['pixel_area']:,} px ({d['area_percent']:.2f}% dari luas gambar)")


In [ ]:
# ==============================================================================
# ⚡ PENGUJIAN PROMPT TERPISAH 4: ROAD CRACK (RETAKAN ASPAL)
# ==============================================================================
TEST_IMG = "testimage3.png"

# Prompt On-Demand untuk Retakan Aspal:
# Retakan memanjang di bagian bawah aspal
CRACK_BOXES = [[53, 495, 360, 573]]
CRACK_POINTS = [[180, 535], [280, 550]]

print("Menjalankan Pure SAM untuk prompt: 'Road Crack (Retakan Jalan)'...")
overlay_crack, dets_crack, _ = run_pure_sam_prompt(
    image_source=TEST_IMG,
    sam_net=sam_model,
    bboxes=CRACK_BOXES,
    points=CRACK_POINTS,
    color=(255, 0, 0),      # Biru (BGR)
    prompt_name="Crack"
)

# Visualisasi Side-by-Side
res_rgb = cv2.cvtColor(overlay_crack, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_orig)
axes[0].set_title(f"Gambar Asli: {TEST_IMG}", fontsize=12)
axes[0].axis("off")

axes[1].imshow(res_rgb)
axes[1].set_title(f"Hasil Segmentasi Pure SAM: Crack ({len(dets_crack)} segmen)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("📊 Ringkasan Metrik Crack:")
for d in dets_crack:
    print(f"  • Segmen {d['mask_id']}: Luas {d['pixel_area']:,} px ({d['area_percent']:.2f}% dari luas gambar)")


## 5. Unified Multi-Prompt Pipeline (Penggabungan Seluruh Prompt Menjadi Satu)

Setelah setiap kategori diuji secara terpisah, sekarang kita menggabungkan seluruh target prompt (**Pothole, Manhole, Road Crack, Sign**) ke dalam satu eksekusi terpadu.

### 🎨 Palet Warna Terstandarisasi:
- 🔴 **Pothole (Lubang Jalan)**: `(0, 0, 255)` (Merah)
- 🟢 **Manhole (Tutup Got)**: `(0, 255, 0)` (Hijau)
- 🔵 **Road Crack (Retakan Aspal)**: `(255, 0, 0)` (Biru)
- 🟡 **Traffic Sign (Rambu Jalan)**: `(0, 255, 255)` (Kuning)

In [ ]:
# Konfigurasi Seluruh Target Prompt Gabungan
UNIFIED_TARGETS = {
    "Pothole": {
        "color": (0, 0, 255),      # Merah
        "boxes": [[424, 323, 720, 454], [200, 375, 266, 412]],
        "points": [[560, 385], [230, 395]]
    },
    "Manhole": {
        "color": (0, 255, 0),      # Hijau
        "boxes": [[78, 376, 179, 414]],
        "points": [[128, 395]]
    },
    "Crack": {
        "color": (255, 0, 0),      # Biru
        "boxes": [[53, 495, 360, 573]],
        "points": [[180, 535]]
    }
}

IMAGE_SAMPLE = "testimage3.png"
img_base = cv2.imread(IMAGE_SAMPLE)
combined_overlay = img_base.copy()
unified_metrics_list = []

print(f"Menjalankan Unified Multi-Prompt Pipeline pada: {IMAGE_SAMPLE}...")

for category_name, cfg in UNIFIED_TARGETS.items():
    ov_cat, dets_cat, _ = run_pure_sam_prompt(
        image_source=img_base,
        sam_net=sam_model,
        bboxes=cfg.get("boxes"),
        points=cfg.get("points"),
        color=cfg["color"],
        prompt_name=category_name
    )
    # Gabungkan mask ke overlay utama
    mask_diff = cv2.absdiff(ov_cat, img_base)
    mask_gray = cv2.cvtColor(mask_diff, cv2.COLOR_BGR2GRAY)
    active_mask = mask_gray > 5
    combined_overlay[active_mask] = ov_cat[active_mask]
    unified_metrics_list.extend(dets_cat)

# Visualisasi Komparasi Side-by-Side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(cv2.cvtColor(img_base, cv2.COLOR_BGR2RGB))
axes[0].set_title("1. Citra Asli Permukaan Jalan", fontsize=12)
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(combined_overlay, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"2. Hasil Unified Pure SAM Multi-Prompt ({len(unified_metrics_list)} Objek Terdeteksi)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

# Simpan hasil gabungan
cv2.imwrite("hasil_unified_pure_sam.jpg", combined_overlay)
print("✅ Hasil segmentasi gabungan disimpan ke: 'hasil_unified_pure_sam.jpg'")

# ==============================================================================
# 📊 TABEL REKAPITULASI METRIK KEPARAHAN JALAN (HIBAH VLM)
# ==============================================================================
total_img_px = img_base.shape[0] * img_base.shape[1]
summary_table = {}

for d in unified_metrics_list:
    p = d["prompt"]
    if p not in summary_table:
        summary_table[p] = {"count": 0, "total_pixels": 0}
    summary_table[p]["count"] += 1
    summary_table[p]["total_pixels"] += d["pixel_area"]

print("\n" + "=" * 65)
print("📑 LAPORAN METRIK SEGMENTASI MULTI-PROMPT GABUNGAN (HIBAH VLM)")
print("=" * 65)
print(f"{'Kategori Objek':<18} | {'Jumlah Objek':<14} | {'Total Luas (px)':<16} | {'Rasio Jalan (%)'}")
print("-" * 65)
for p_name, stats in summary_table.items():
    pct = (stats['total_pixels'] / total_img_px) * 100.0
    print(f"{p_name:<18} | {stats['count']:<14} | {stats['total_pixels']:<16,} | {pct:.2f}%")
print("=" * 65)


## 6. Bagian 2: Inferensi Video (`testvideo1.mp4`) dengan Placeholder Kelas Dinamis

Di bagian ini, kita menjalankan inferensi Pure SAM langsung pada rekaman video jalan (**`testvideo1.mp4`**).

### 🎯 Desain Placeholder Target Fleksibel (Pure SAM):
Anda dapat **menambah, menghapus, atau menyesuaikan** kelas prompt apa saja pada dictionary `VIDEO_PROMPT_CONFIG` di bawah ini. Cukup tentukan koordinat area Box atau Point target untuk setiap kategori.

In [ ]:
# ==============================================================================
# 🎯 PLACEHOLDER KONFIGURASI PROMPT PURE SAM UNTUK VIDEO (testvideo1.mp4)
# Bebas menambah, mengubah, atau menyesuaikan prompt untuk kelas apa saja!
# 100% Murni SAM 2.1 / SAM 3.1 tanpa YOLO
# ==============================================================================
VIDEO_PROMPT_CONFIG = {
    "pothole": {
        "label": "Pothole (Lubang)",
        "color": (0, 0, 255),               # Merah (BGR)
        "boxes": [[600, 600, 1100, 950]],   # Box Prompt koordinat area jalan berlubang
        "points": [[850, 750]],             # Point Prompt on-demand
        "point_labels": [1]
    },
    "road crack": {
        "label": "Crack (Retakan Aspal)",
        "color": (255, 0, 0),               # Biru (BGR)
        "boxes": [[350, 500, 650, 750]],
        "points": [[500, 620]],
        "point_labels": [1]
    },
    "manhole": {
        "label": "Manhole (Tutup Got)",
        "color": (0, 255, 0),               # Hijau (BGR)
        "boxes": [[200, 580, 420, 800]],
        "points": [[310, 680]],
        "point_labels": [1]
    },
    "traffic sign": {
        "label": "Traffic Sign (Rambu)",
        "color": (0, 255, 255),             # Kuning (BGR)
        "boxes": [[900, 150, 1150, 380]],
        "points": [[1025, 260]],
        "point_labels": [1]
    },

    # 💡 CONTOH MENAMBAH KELAS BARU (Cukup hilangkan tanda komentar '#' di bawah):
    # "car": {
    #     "label": "Kendaraan / Mobil",
    #     "color": (255, 0, 255),           # Magenta
    #     "boxes": [[1150, 400, 1550, 700]],
    #     "points": [[1350, 550]],
    #     "point_labels": [1]
    # },
    # "pedestrian": {
    #     "label": "Pejalan Kaki",
    #     "color": (255, 255, 0),           # Cyan
    #     "boxes": [[100, 400, 250, 700]],
    #     "points": [[175, 550]],
    #     "point_labels": [1]
    # },
}

print("✅ Konfigurasi Placeholder Video Pure SAM aktif:")
for k, v in VIDEO_PROMPT_CONFIG.items():
    print(f"  • Kelas: '{k}' | Label: '{v['label']}' | Box: {len(v.get('boxes', []))} | Point: {len(v.get('points', []))}")


In [ ]:
import time

def process_video_pure_sam(
    video_path,
    output_path,
    prompt_config,
    sam_net,
    max_frames=90,           # Set ke None jika ingin memproses video penuh
    skip_frames=1,           # 1 = proses setiap frame, 2 = setiap 2 frame (lebih cepat)
    resize_dim=(1280, 720)   # Resolusi pemrosesan untuk efisiensi VRAM & kecepatan FPS
):
    """
    Memproses video uji menggunakan Pure SAM 2.1 Prompt-on-Demand (Tanpa YOLO).
    Dilengkapi HUD telemetry real-time: FPS counter, hitungan segmen per kelas, dan mask transparan.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Video tidak ditemukan: {video_path}")

    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_w, out_h = resize_dim if resize_dim else (orig_w, orig_h)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps_in / skip_frames, (out_w, out_h))

    # Skalakan koordinat prompt jika video di-resize
    sx = out_w / orig_w if resize_dim else 1.0
    sy = out_h / orig_h if resize_dim else 1.0

    scaled_config = {}
    for cat, cfg in prompt_config.items():
        scaled_boxes = []
        for b in cfg.get("boxes", []):
            scaled_boxes.append([int(b[0] * sx), int(b[1] * sy), int(b[2] * sx), int(b[3] * sy)])
        scaled_points = []
        for p in cfg.get("points", []):
            scaled_points.append([int(p[0] * sx), int(p[1] * sy)])

        scaled_config[cat] = {
            "label": cfg["label"],
            "color": cfg["color"],
            "boxes": scaled_boxes,
            "points": scaled_points,
            "point_labels": cfg.get("point_labels", [1] * len(scaled_points))
        }

    print("=" * 60)
    print(f"🎬 MEMULAI PEMROSESAN VIDEO PURE SAM: {video_path}")
    print(f"Resolusi Input : {orig_w}x{orig_h} @ {fps_in:.1f} FPS")
    print(f"Resolusi Output: {out_w}x{out_h}")
    print(f"Target Kelas   : {list(prompt_config.keys())}")
    print(f"Batas Frame    : {max_frames if max_frames else 'Seluruh Frame (' + str(total_frames) + ')'}")
    print("=" * 60)

    frame_idx = 0
    processed_count = 0
    start_time = time.time()

    temporal_damage_log = []
    sample_frames_display = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % skip_frames != 0:
            frame_idx += 1
            continue

        if max_frames and processed_count >= max_frames:
            break

        t_frame_start = time.time()

        if resize_dim:
            frame_proc = cv2.resize(frame, resize_dim)
        else:
            frame_proc = frame

        annotated_frame = frame_proc.copy()
        frame_counts = {cat: 0 for cat in prompt_config.keys()}
        total_frame_dmg_px = 0

        # Eksekusi prompt per kategori pada frame ini
        for cat_name, cat_cfg in scaled_config.items():
            b_list = cat_cfg.get("boxes", [])
            p_list = cat_cfg.get("points", [])
            if not b_list and not p_list:
                continue

            ov_res, dets, _ = run_pure_sam_prompt(
                image_source=frame_proc,
                sam_net=sam_net,
                bboxes=b_list if b_list else None,
                points=p_list if p_list else None,
                labels=cat_cfg.get("point_labels"),
                color=cat_cfg["color"],
                prompt_name=cat_cfg["label"]
            )

            # Blending mask kategori ke frame utama
            diff = cv2.absdiff(ov_res, frame_proc)
            gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
            active = gray > 5
            annotated_frame[active] = ov_res[active]

            frame_counts[cat_name] = len(dets)
            for d in dets:
                total_frame_dmg_px += d["pixel_area"]

        t_frame_end = time.time()
        curr_fps = 1.0 / max(1e-5, (t_frame_end - t_frame_start))

        # ======================================================================
        # RENDER HUD DASHBOARD OVERLAY PADA VIDEO
        # ======================================================================
        hud_bg = annotated_frame.copy()
        cv2.rectangle(hud_bg, (10, 10), (430, 75 + len(prompt_config) * 22), (20, 20, 20), -1)
        annotated_frame = cv2.addWeighted(annotated_frame, 0.25, hud_bg, 0.75, 0)

        cv2.putText(annotated_frame, f"Pure SAM 2.1 Video Inspection | Frame: {frame_idx}/{total_frames}", 
                    (20, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        cv2.putText(annotated_frame, f"Kecepatan: {curr_fps:.1f} FPS | Durasi: {(frame_idx / fps_in):.2f}s", 
                    (20, 54), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 255, 200), 1)

        y_offset = 78
        for cat_name, cat_cfg in prompt_config.items():
            col = cat_cfg["color"]
            disp_label = cat_cfg["label"]
            cnt = frame_counts.get(cat_name, 0)
            cv2.circle(annotated_frame, (28, y_offset - 4), 6, col, -1)
            cv2.putText(annotated_frame, f"{disp_label}: {cnt}", 
                        (42, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1)
            y_offset += 22

        writer.write(annotated_frame)

        # Catat metrik temporal
        temporal_damage_log.append({
            "frame": frame_idx,
            "time_sec": frame_idx / fps_in,
            "counts": frame_counts,
            "damage_pixels": total_frame_dmg_px,
            "damage_pct": (total_frame_dmg_px / (out_w * out_h)) * 100.0
        })

        # Simpan sampel galeri
        if processed_count in [0, max_frames // 4, max_frames // 2, (max_frames * 3) // 4, max_frames - 1]:
            sample_frames_display.append((frame_idx, annotated_frame.copy()))

        processed_count += 1
        frame_idx += 1

        if processed_count % 15 == 0 or processed_count == max_frames:
            elapsed = time.time() - start_time
            print(f"  ⚡ Diproses: {processed_count}/{max_frames or total_frames} frames | Kecepatan rata-rata: {processed_count/elapsed:.1f} FPS")

    cap.release()
    writer.release()

    total_time = time.time() - start_time
    print("=" * 60)
    print("✅ PEMROSESAN VIDEO PURE SAM SELESAI!")
    print(f"Total Frame Diproses: {processed_count} frames")
    print(f"Total Waktu          : {total_time:.2f} detik ({processed_count/total_time:.1f} FPS rata-rata)")
    print(f"File Hasil Disimpan  : {output_path}")
    print("=" * 60)

    return temporal_damage_log, sample_frames_display


In [ ]:
# ==============================================================================
# 🚀 EKSEKUSI INFERENSI VIDEO PURE SAM PADA 'testvideo1.mp4'
# ==============================================================================
INPUT_VIDEO = "testvideo1.mp4"
OUTPUT_VIDEO = "hasil_video_pure_sam.mp4"

# Catatan: Atur max_frames=90 (sekitar 3 detik) untuk verifikasi cepat.
# Ubah max_frames=None jika ingin memproses keseluruhan video.
damage_log, sample_frames = process_video_pure_sam(
    video_path=INPUT_VIDEO,
    output_path=OUTPUT_VIDEO,
    prompt_config=VIDEO_PROMPT_CONFIG,
    sam_net=sam_model,
    max_frames=90,           # Ganti ke None untuk full video
    skip_frames=1,
    resize_dim=(1280, 720)
)

# Visualisasi Galeri Sampel Frame Hasil Video
if sample_frames:
    num_samples = len(sample_frames)
    fig, axes = plt.subplots(1, num_samples, figsize=(5 * num_samples, 4))
    if num_samples == 1:
        axes = [axes]

    for ax, (f_idx, f_img) in zip(axes, sample_frames):
        ax.imshow(cv2.cvtColor(f_img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame ke-{f_idx}", fontsize=11)
        ax.axis("off")

    plt.suptitle("📸 Galeri Cuplikan Inferensi Video (Pure SAM 2.1)", fontsize=14, y=1.03)
    plt.tight_layout()
    plt.show()


## 7. Analisis Metrik Temporal Kerusakan Jalan (Hibah VLM Report)

Grafik fluktuasi area kerusakan jalan sepanjang durasi video untuk mengidentifikasi titik jalan dengan tingkat keparahan tertinggi (*critical road segments*).

In [ ]:
if damage_log:
    times = [d["time_sec"] for d in damage_log]
    dmg_pcts = [d["damage_pct"] for d in damage_log]
    pothole_counts = [d["counts"].get("pothole", 0) for d in damage_log]
    crack_counts = [d["counts"].get("road crack", 0) for d in damage_log]

    fig, ax1 = plt.subplots(figsize=(14, 5))

    color_dmg = 'tab:red'
    ax1.set_xlabel("Waktu Video (detik)", fontsize=11)
    ax1.set_ylabel("Rasio Luas Kerusakan Jalan (%)", color=color_dmg, fontsize=11)
    line1 = ax1.plot(times, dmg_pcts, color=color_dmg, linewidth=2, label="Rasio Luas Kerusakan (%)")
    ax1.tick_params(axis='y', labelcolor=color_dmg)
    ax1.grid(True, linestyle="--", alpha=0.5)

    ax2 = ax1.twinx()
    color_cnt = 'tab:blue'
    ax2.set_ylabel("Jumlah Segmen Objek", color=color_cnt, fontsize=11)
    line2 = ax2.plot(times, pothole_counts, color="red", linestyle=":", label="Segmen Pothole")
    line3 = ax2.plot(times, crack_counts, color="blue", linestyle="--", label="Segmen Crack")
    ax2.tick_params(axis='y', labelcolor=color_cnt)

    lines = line1 + line2 + line3
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc="upper left")

    plt.title("📈 Fluktuasi Temporal Kerusakan Permukaan Jalan Sepanjang Video (Pure SAM Hibah VLM)", fontsize=13)
    plt.tight_layout()
    plt.savefig("grafik_analisis_temporal_jalan.png", dpi=300)
    plt.show()

    max_idx = int(np.argmax(dmg_pcts))
    print("=" * 60)
    print("📊 KESIMPULAN ANALISIS TEMPORAL:")
    print(f"  • Puncak Kerusakan Tertinggi : Frame ke-{damage_log[max_idx]['frame']} (Detik ke-{times[max_idx]:.2f})")
    print(f"  • Persentase Kerusakan Maks  : {dmg_pcts[max_idx]:.2f}% dari luas jalan")
    print(f"  • Rekapitulasi Deteksi Maks  : {damage_log[max_idx]['counts']}")
    print("=" * 60)
